# Lección 04 — Herramientas (Tool Use)

En este notebook vas a aprender a:
1. Definir herramientas correctamente con buenos schemas
2. Combinar múltiples herramientas en un agente
3. Manejar errores cuando una herramienta falla
4. Pedir confirmación antes de ejecutar acciones importantes

In [ ]:
%pip install anthropic python-dotenv -q

In [ ]:
import anthropic
import json
from datetime import datetime
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
print("Setup listo.")

## Parte 1 — Definir herramientas con buenos schemas

Un buen schema tiene:
- **Nombre** descriptivo (snake_case)
- **Descripción** que explica cuándo usarla (no solo qué hace)
- **Parámetros** con tipo y descripción claros
- **required** — lista de los parámetros obligatorios

Vamos a definir 3 herramientas para un agente de viajes completo.

In [ ]:
# --- FUNCIONES PYTHON ---

def buscar_destinos(tipo_clima: str = None, presupuesto_max_usd: int = None) -> list:
    """Busca destinos según filtros."""
    destinos = [
        {"nombre": "Cancún", "clima": "tropical", "precio_semana": 1100, "pais": "México"},
        {"nombre": "Cartagena", "clima": "tropical", "precio_semana": 800, "pais": "Colombia"},
        {"nombre": "Buenos Aires", "clima": "templado", "precio_semana": 900, "pais": "Argentina"},
        {"nombre": "Barcelona", "clima": "mediterráneo", "precio_semana": 1800, "pais": "España"},
        {"nombre": "Tokio", "clima": "variado", "precio_semana": 2200, "pais": "Japón"},
        {"nombre": "Medellín", "clima": "primaveral", "precio_semana": 700, "pais": "Colombia"},
    ]
    resultados = destinos
    if tipo_clima:
        resultados = [d for d in resultados if tipo_clima.lower() in d["clima"].lower()]
    if presupuesto_max_usd:
        resultados = [d for d in resultados if d["precio_semana"] <= presupuesto_max_usd]
    return resultados


def verificar_vuelos(origen: str, destino: str, fecha_salida: str) -> dict:
    """Verifica disponibilidad de vuelos entre dos ciudades."""
    # Simulación de búsqueda de vuelos
    vuelos_simulados = {
        ("Buenos Aires", "Cancún"): {"disponible": True, "precio_usd": 650, "duracion": "8h 30min", "escalas": 1},
        ("Buenos Aires", "Cartagena"): {"disponible": True, "precio_usd": 480, "duracion": "6h", "escalas": 0},
        ("Buenos Aires", "Barcelona"): {"disponible": True, "precio_usd": 890, "duracion": "13h", "escalas": 0},
        ("Buenos Aires", "Medellín"): {"disponible": True, "precio_usd": 420, "duracion": "5h", "escalas": 0},
        ("Buenos Aires", "Tokio"): {"disponible": False, "motivo": "sin vuelos disponibles para esa fecha"},
    }
    clave = (origen, destino)
    resultado = vuelos_simulados.get(clave, {"disponible": False, "motivo": "ruta no encontrada"})
    resultado["fecha_consulta"] = fecha_salida
    return resultado


# Esta es una herramienta de ACCIÓN — simula una reserva real
reservas_realizadas = []  # guardamos las reservas para el demo

def reservar_viaje(destino: str, fecha_salida: str, pasajeros: int, presupuesto_usd: int) -> dict:
    """Realiza la reserva de un viaje. ACCIÓN IRREVERSIBLE."""
    reserva = {
        "id_reserva": f"POL-{len(reservas_realizadas)+1001}",
        "destino": destino,
        "fecha_salida": fecha_salida,
        "pasajeros": pasajeros,
        "total_usd": presupuesto_usd,
        "estado": "confirmada",
        "fecha_reserva": datetime.now().strftime("%Y-%m-%d %H:%M")
    }
    reservas_realizadas.append(reserva)
    return reserva


# --- SCHEMAS PARA CLAUDE ---

schemas = [
    {
        "name": "buscar_destinos",
        "description": "Busca destinos de viaje disponibles. Usá esta herramienta cuando el usuario quiera ver opciones de viaje o filtrar por clima y presupuesto.",
        "input_schema": {
            "type": "object",
            "properties": {
                "tipo_clima": {"type": "string", "description": "Filtrar por clima: tropical, templado, mediterráneo, variado, primaveral"},
                "presupuesto_max_usd": {"type": "integer", "description": "Presupuesto máximo en dólares por semana"}
            }
        }
    },
    {
        "name": "verificar_vuelos",
        "description": "Verifica disponibilidad y precios de vuelos entre dos ciudades. Usá esta herramienta antes de confirmar una recomendación.",
        "input_schema": {
            "type": "object",
            "properties": {
                "origen": {"type": "string", "description": "Ciudad de origen (ej: Buenos Aires)"},
                "destino": {"type": "string", "description": "Ciudad de destino"},
                "fecha_salida": {"type": "string", "description": "Fecha de salida en formato YYYY-MM-DD"}
            },
            "required": ["origen", "destino", "fecha_salida"]
        }
    },
    {
        "name": "reservar_viaje",
        "description": "Confirma y realiza la reserva de un viaje. SOLO usá esta herramienta si el usuario confirmó explícitamente que quiere reservar.",
        "input_schema": {
            "type": "object",
            "properties": {
                "destino": {"type": "string", "description": "Ciudad de destino"},
                "fecha_salida": {"type": "string", "description": "Fecha de salida en formato YYYY-MM-DD"},
                "pasajeros": {"type": "integer", "description": "Cantidad de pasajeros"},
                "presupuesto_usd": {"type": "integer", "description": "Presupuesto total en dólares"}
            },
            "required": ["destino", "fecha_salida", "pasajeros", "presupuesto_usd"]
        }
    }
]

funciones = {
    "buscar_destinos": buscar_destinos,
    "verificar_vuelos": verificar_vuelos,
    "reservar_viaje": reservar_viaje
}

print(f"3 herramientas definidas: {[s['name'] for s in schemas]}")

## Parte 2 — Agente con múltiples herramientas

Ahora creamos el agente completo que puede usar las 3 herramientas y vemos cómo las combina para responder una consulta.

In [ ]:
def ejecutar_agente_viajes(mensaje_usuario: str, historial: list = None) -> tuple:
    """Ejecuta el agente de viajes con soporte para múltiples herramientas."""
    
    if historial is None:
        historial = []
    
    historial.append({"role": "user", "content": mensaje_usuario})
    print(f"Vos: {mensaje_usuario}")
    print("-" * 60)
    
    system = """Sos un agente de viajes completo. Para cada consulta:
1. Buscá destinos disponibles según las preferencias
2. Verificá vuelos para las mejores opciones
3. Presentá un resumen claro con precios totales (vuelo + alojamiento)
4. Si el usuario quiere reservar, pedile CONFIRMACIÓN EXPLÍCITA antes de usar reservar_viaje

Contexto: el usuario viaja siempre desde Buenos Aires.
Fecha actual: 2026-06-01. Las fechas de viaje suelen ser 2-4 semanas después."""
    
    tools_usadas = []
    
    while True:
        respuesta = client.messages.create(
            model="claude-opus-4-5",
            max_tokens=1500,
            system=system,
            tools=schemas,
            messages=historial
        )
        
        if respuesta.stop_reason == "tool_use":
            historial.append({"role": "assistant", "content": respuesta.content})
            resultados_tools = []
            
            for bloque in respuesta.content:
                if bloque.type == "tool_use":
                    print(f"  [Herramienta: {bloque.name}({json.dumps(bloque.input, ensure_ascii=False)})]")
                    tools_usadas.append(bloque.name)
                    
                    fn = funciones[bloque.name]
                    try:
                        resultado = fn(**bloque.input) if bloque.input else fn()
                    except Exception as e:
                        resultado = {"error": str(e)}
                    
                    resultados_tools.append({
                        "type": "tool_result",
                        "tool_use_id": bloque.id,
                        "content": json.dumps(resultado, ensure_ascii=False)
                    })
            
            historial.append({"role": "user", "content": resultados_tools})
            continue
        
        texto_final = next(b.text for b in respuesta.content if b.type == "text")
        historial.append({"role": "assistant", "content": texto_final})
        print(f"Agente: {texto_final}")
        return texto_final, historial


# Prueba 1: búsqueda con filtros
historial_conversacion = []
_, historial_conversacion = ejecutar_agente_viajes(
    "Busco una semana de playa tropical con presupuesto de 1500 dólares todo incluido",
    historial_conversacion
)

In [ ]:
# El agente recuerda el contexto — continuamos la conversación
print("\n" + "="*60)
_, historial_conversacion = ejecutar_agente_viajes(
    "Me gusta Cartagena. ¿Podés verificar vuelos para el 20 de junio?",
    historial_conversacion
)

## Parte 3 — Confirmación antes de acciones irreversibles

Fijate cómo el agente pide confirmación antes de usar `reservar_viaje`.
Esto es una buena práctica para cualquier acción que no se pueda deshacer.

In [ ]:
print("\n" + "="*60)
_, historial_conversacion = ejecutar_agente_viajes(
    "Perfecto, reservá para 2 personas. Confirmo la reserva.",
    historial_conversacion
)

# Ver reservas realizadas
if reservas_realizadas:
    print("\n--- Reserva registrada ---")
    print(json.dumps(reservas_realizadas[-1], indent=2, ensure_ascii=False))

## Resumen

En esta lección trabajaste con herramientas reales:

| Concepto | Lo que aprendiste |
|---|---|
| Schema de tool | Nombre + descripción precisa + parámetros tipados |
| Múltiples tools | El agente elige cuál usar según el contexto |
| Tools en paralelo | Claude puede llamar varias tools en una misma respuesta |
| Manejo de errores | Capturar excepciones y devolverlas como `tool_result` |
| Confirmación | Pedir explicitamente antes de ejecutar acciones irreversibles |

---
En la **Lección 05** vamos a darle **memoria persistente** al agente para que recuerde información entre sesiones.